# WM 2026 – Team-Reiserouten & Reisebelastung

Berechnet pro Team: Reisedistanz, weiteste Einzelreise, Zeitzonen-Belastung und
durchschnittliche Tage zwischen den Spielen.

**Reisemodell (Sternmodell):** Das Team wohnt im **Quartier** und fliegt zu jedem
Spiel hin und zurück:

> `Total_Distance_KM = Σ über alle Spiele:  2 × Luftlinie(Quartier → Spielort)`

Zusätzlich `Chain_Distance_KM` (naive Stadion→Stadion-Kette ohne Quartier) als Vergleich.

## Benötigte Input-CSVs in `data/`

| Datei | Spalten |
|---|---|
| `wm2026_stadien_koordinaten_zeitzonen.csv` | `Stadium_Name, City, Country, Latitude, Longitude, Timezone_UTC, Matches_Count` |
| `spielplan.csv` | `Date, Stadium, Team_A, Team_B` (Date = `JJJJ-MM-TT`, Stadium exakt wie Stadien-CSV) |
| `quartiere.csv` | `Team, BaseCamp_City, Latitude, Longitude` (Team exakt wie im Spielplan) |

**Output:** `data/wm2026_team_reiserouten_distanzen.csv`

> Distanzen sind Luftlinie (Haversine) – Untergrenze. Quartier-Zeitzone wird aus dem
> nächstgelegenen Stadion abgeleitet.

In [ ]:
import warnings
from math import radians, sin, cos, sqrt, asin
from datetime import datetime
from pathlib import Path

import pandas as pd


def haversine_distance(lat1, lon1, lat2, lon2):
    """Luftlinien-Distanz zwischen zwei Koordinaten in km."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return R * 2 * asin(sqrt(a))


def find_path(filename):
    """Findet eine Datei egal ob aus Projekt-Root oder notebooks/ ausgeführt."""
    for base in (Path("data"), Path("../data"), Path(".")):
        if (base / filename).exists():
            return base / filename
    return Path("../data") / filename


STADIEN_CSV = find_path("wm2026_stadien_koordinaten_zeitzonen.csv")
SPIELPLAN_CSV = find_path("spielplan.csv")
QUARTIERE_CSV = find_path("quartiere.csv")
OUTPUT_CSV = STADIEN_CSV.parent / "wm2026_team_reiserouten_distanzen.csv"

In [ ]:
# 1. Stadien laden (utf-8-sig entfernt das BOM am Dateianfang)
df_stadien = pd.read_csv(STADIEN_CSV, encoding="utf-8-sig")
stadien_dict = {
    r["Stadium_Name"]: {
        "Latitude": r["Latitude"], "Longitude": r["Longitude"],
        "Timezone_UTC": r["Timezone_UTC"], "City": r["City"],
    }
    for _, r in df_stadien.iterrows()
}


def naechste_stadion_tz(lat, lon):
    """Zeitzone des nächstgelegenen Stadions (für das Quartier)."""
    nearest = min(stadien_dict.values(),
                  key=lambda s: haversine_distance(lat, lon, s["Latitude"], s["Longitude"]))
    return nearest["Timezone_UTC"]


print(f"{len(stadien_dict)} Stadien geladen.")

In [ ]:
# 2. Spielplan + Quartiere laden
df_spielplan = pd.read_csv(SPIELPLAN_CSV, encoding="utf-8-sig")
pflicht = {"Date", "Stadium", "Team_A", "Team_B"}
assert not (pflicht - set(df_spielplan.columns)), f"Spielplan fehlen Spalten: {pflicht - set(df_spielplan.columns)}"
# Leerzeichen in Team-/Stadionnamen trimmen (verhindert stille Mismatches)
for c in ("Team_A", "Team_B", "Stadium"):
    df_spielplan[c] = df_spielplan[c].astype(str).str.strip()
print(f"Spielplan: {len(df_spielplan)} Spiele.")

df_quartiere = pd.read_csv(QUARTIERE_CSV, encoding="utf-8-sig")
df_quartiere = df_quartiere.loc[:, ~df_quartiere.columns.str.startswith("Unnamed")]  # leere Spalte durch End-Komma weg
df_quartiere["Team"] = df_quartiere["Team"].astype(str).str.strip()
pflicht_q = {"Team", "BaseCamp_City", "Latitude", "Longitude"}
assert not (pflicht_q - set(df_quartiere.columns)), f"Quartiere fehlen Spalten: {pflicht_q - set(df_quartiere.columns)}"

quartier_dict = {
    r["Team"]: {
        "Latitude": r["Latitude"], "Longitude": r["Longitude"],
        "City": r["BaseCamp_City"],
        "Timezone_UTC": naechste_stadion_tz(r["Latitude"], r["Longitude"]),
    }
    for _, r in df_quartiere.iterrows()
}
print(f"Quartiere: {len(quartier_dict)} Teams.")

In [ ]:
def berechne_team_reiseroute(team_name, df_spielplan, stadien_dict, quartier_dict):
    """Reise-Kennzahlen im Sternmodell (Quartier → Spielort → zurück)."""
    # Team kann Team_A ODER Team_B sein – beide Seiten berücksichtigen
    maske = (df_spielplan["Team_A"] == team_name) | (df_spielplan["Team_B"] == team_name)
    spiele = df_spielplan[maske].sort_values("Date").reset_index(drop=True)
    if len(spiele) == 0:
        return None

    base = quartier_dict.get(team_name)
    if base is None:
        warnings.warn(f"{team_name}: kein Quartier gefunden – übersprungen.")
        return None

    fehlende = []

    def lookup(name):
        info = stadien_dict.get(name)
        if info is None:
            fehlende.append(name)
        return info

    total_base = 0.0     # Sternmodell: Σ 2×(Quartier→Spielort)
    longest_trip = 0.0   # weiteste einzelne Anreise Quartier→Spielort
    total_chain = 0.0    # Vergleich: naive Stadion→Stadion-Kette
    tz_shift = 0         # Σ |Spielort_tz − Quartier_tz|
    max_tz_diff = 0
    prev = None

    for i in range(len(spiele)):
        cur = lookup(spiele.loc[i, "Stadium"])
        if cur is None:
            prev = None  # Kette unterbrochen, kein Crash
            continue

        d_base = haversine_distance(base["Latitude"], base["Longitude"], cur["Latitude"], cur["Longitude"])
        total_base += 2 * d_base
        longest_trip = max(longest_trip, d_base)

        tz = abs(cur["Timezone_UTC"] - base["Timezone_UTC"])
        tz_shift += tz
        max_tz_diff = max(max_tz_diff, tz)

        if prev is not None:
            total_chain += haversine_distance(prev["Latitude"], prev["Longitude"], cur["Latitude"], cur["Longitude"])
        prev = cur

    dates = [datetime.strptime(str(d), "%Y-%m-%d") for d in spiele["Date"]]
    gaps = [(dates[i] - dates[i - 1]).days for i in range(1, len(dates))]

    if fehlende:
        warnings.warn(f"{team_name}: Stadion nicht in Stadien-CSV: {sorted(set(fehlende))}")

    return {
        "Team_Name": team_name,
        "BaseCamp_City": base["City"],
        "Games_Count": len(spiele),
        "Total_Distance_KM": round(total_base),     # Sternmodell (Hauptmetrik)
        "Longest_Trip_KM": round(longest_trip),     # weiteste einzelne Anreise
        "Chain_Distance_KM": round(total_chain),    # naiver Vergleich ohne Quartier
        "Timezone_Shift_Total_H": tz_shift,         # Summe Zeitzonendiff. ggü. Quartier
        "Max_Timezone_Diff_H": max_tz_diff,
        "Avg_Days_Between_Games": round(sum(gaps) / len(gaps), 1) if gaps else 0,
    }

In [ ]:
# 3. Alle Teams aggregieren
alle_teams = sorted(set(df_spielplan["Team_A"]) | set(df_spielplan["Team_B"]))
team_routen = [
    r for t in alle_teams
    if (r := berechne_team_reiseroute(t, df_spielplan, stadien_dict, quartier_dict))
]

df_team_routen = (
    pd.DataFrame(team_routen)
    .sort_values("Total_Distance_KM", ascending=False)
    .reset_index(drop=True)
)

df_team_routen.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"{len(df_team_routen)} Teams aggregiert – gespeichert: {OUTPUT_CSV}")

In [ ]:
# 4. Top 10 nach Reisedistanz
df_team_routen.head(10)

In [ ]:
# 5. Reiselinien-Tabelle (für Karte in Tableau via MAKELINE)
# Breites Format: eine Zeile pro Etappe (Quartier -> Spielort) mit Start-/Ziel-Koordinaten.
# In Tableau:  MAKELINE( MAKEPOINT([Quartier_Lat],[Quartier_Lon]),
#                        MAKEPOINT([Spielort_Lat],[Spielort_Lon]) )
LINIEN_CSV = STADIEN_CSV.parent / "wm2026_reiselinien.csv"

leg_rows = []
for team in alle_teams:
    base = quartier_dict.get(team)
    if base is None:
        continue
    maske = (df_spielplan["Team_A"] == team) | (df_spielplan["Team_B"] == team)
    spiele = df_spielplan[maske].sort_values("Date").reset_index(drop=True)
    for _, r in spiele.iterrows():
        s = stadien_dict.get(r["Stadium"])
        if s is None:
            continue
        leg_rows.append({
            "Team": team,
            "BaseCamp_City": base["City"],
            "Spielort": r["Stadium"],
            "Spiel_Datum": r["Date"],
            "Quartier_Lat": base["Latitude"], "Quartier_Lon": base["Longitude"],
            "Spielort_Lat": s["Latitude"], "Spielort_Lon": s["Longitude"],
            "Distanz_KM": round(haversine_distance(base["Latitude"], base["Longitude"], s["Latitude"], s["Longitude"])),
        })

df_legs = pd.DataFrame(leg_rows)
df_legs.to_csv(LINIEN_CSV, index=False, encoding="utf-8-sig")
print(f"{len(df_legs)} Etappen – gespeichert: {LINIEN_CSV}")
df_legs.head(6)